# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhruvaiyer2-netizen/FlyRankMLProject/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane involves ranking - I'm scoring pages by how far their CTR falls below what's expected for their position tier, and sorting the results, rather than sorting pages into fixed categories.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

What I'd predict is an opportunity score — how far a page's CTR falls below the median CTR for its position tier, among pages with reliable volume (>= 500 impressions). This is a proxy, not an observed outcome: there's no column in this data recording "this page was a true missed opportunity" or "editing this page increased its clicks." I defined the label myself, using a rule (tier-relative gap) that I chose and validated by checking the distribution made sense — not something the data told me directly.
A genuine observed-outcome version of this label would require something like: pages where an edit was actually made, paired with their CTR change in the following period. I don't have that here, so any claim I make has to stay at the level of "this page's current CTR looks anomalous for its position" — not "this page's CTR will improve" or "this page was mishandled."

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I'll use precision@50: of my top 50 ranked candidates (largest tier-relative CTR gap, filtered to >= 500 impressions), what fraction hold up as genuine opportunities on manual review — meaning the page really is well-positioned (top 20) with CTR clearly below its tier's typical level, not an artifact of a tiny sub-population or a data quirk. "Good" means a majority of my top 50 survive that manual check — I'd want to see something clearly above chance (the guide's own baseline example hit 0.240; I'd want to at least match or beat that with my rule, and use it as the bar any future model has to clear).
I'm choosing precision@K over something like overall accuracy because my output is a ranked list for limited reviewer time — nobody reviews all 16,726 pages, so what matters is whether the top of the list is trustworthy, not whether I've correctly labeled every single row.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page, summarized over the trailing 90-day period. All finer details have been aggregated into page level totals, like total impressions_90d, or rates like ctr, avg position before reaching the table. We can confirm this by checking content_id uniqueness: 16,726 rows and 16,726 unique content_id values, so there are no duplicates or mismatches in row counts.
Looking at the sample row chosen, we have content content_304f48230142 which is has an average position of 10.6 with a CTR of 0.76, which means it falls in the 4-10 position tier, and has a CTR more than triple of what it is expected to have (0.23), proving that my pipeline doesn't just flag everything, it seperates strong performers from those with genuine gaps.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# One row = one page, summarized over its own 90-day window
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

volume_filtered = df[df["impressions_90d"] >= 500]
sample_page = volume_filtered.iloc[0]
print(sample_page[["content_id", "avg_position", "impressions_90d", "ctr", "content_age_days"]])
print(f"\nTotal rows in my working slice: {len(volume_filtered)}")
print(f"Unique content_ids: {volume_filtered['content_id'].nunique()}")

content_id          content_304f48230142
avg_position                        10.6
impressions_90d                     3803
ctr                                 0.76
content_age_days                     187
Name: 0, dtype: object

Total rows in my working slice: 16726
Unique content_ids: 16726


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The pattern is too messy for a simple if-statement because position alone cannot define the expected CTR. From the previous notebook, I've tested and discovered that the median CTR was almost identical between the position tiers 1-3 and 4-10, with a wide spread in the position tier 1-3. This tells me that some other factor or combination of factors, such as content age, intent type, or even something not present in the dataset is accounting for the variance, which cannot be accurately represented using a combination of if-statements. A fixed rule also cannot account properly for how the volume of impresssions affects the level of trust that can be placed in the gap. A page with 3000 impressions having a gap is more trustworthy compared to a page with 500 impressions. An if-statement can encode one or two conditions well, but when the requirement involves multiple conditions, and the interaction of various signals and factors, a model is better suited to learn the complete picture, involving the weighted combinations of the signals better than a simple fixed rule which would require constant fine-tuning using trial and error.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.